In [1]:
# Imports and setup
import sys
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add workspace root to Python path for imports
workspace_root = Path('/workspace-hku/hku-datawork')
if str(workspace_root) not in sys.path:
    sys.path.insert(0, str(workspace_root))

# Try to use RAPIDS for speed, fallback to pandas/numpy
try:
    import cudf
    import cupy as cp
    RAPIDS_AVAILABLE = True
    print("✓ RAPIDS available - using GPU acceleration")
except ImportError:
    RAPIDS_AVAILABLE = False
    print("⚠ RAPIDS not available - using CPU (pandas/numpy)")

import pandas as pd
import numpy as np
import logging
from tqdm import tqdm

# Import cointegration modules
from hypothesis_testing.cointegration import (
    load_price_data,
    generate_baskets_clustering,
    test_basket_cointegration,
    test_persistence_rolling,
    compute_max_zscore,
    optimize_lookback_window,
    optimize_basket_size,
    plot_spread_analysis,
    plot_lookback_optimization,
    print_summary_statistics,
    test_baskets_parallel
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

DATA_DIR = Path("../../../hku-data/test_data")


✓ RAPIDS available - using GPU acceleration


In [2]:
# Phase 1: Load price data
price_data = load_price_data(DATA_DIR, years_back=1.0)
symbols = [col.replace('_close', '') for col in price_data.columns]
print(f"\n✓ Data loaded: {len(symbols)} symbols, {len(price_data)} time points")


2025-10-31 11:10:25,940 - INFO - Loading data from 116 CSV files...
Loading CSVs: 100%|██████████| 116/116 [00:16<00:00,  7.04it/s]
2025-10-31 11:10:42,431 - INFO - Aligning timestamps across all symbols...
2025-10-31 11:10:42,992 - INFO - Loaded 30335 timestamps for 106 symbols
2025-10-31 11:10:42,994 - INFO - Date range: 2024-10-31 11:15:00+00:00 to 2025-09-12 10:45:00+00:00



✓ Data loaded: 106 symbols, 30335 time points


In [ ]:
# Phase 2 & 3: Generate baskets and test initial cointegration
logger.info("Generating candidate baskets...")
candidate_baskets = generate_baskets_clustering(price_data, n_clusters=8, 
                                                 min_basket_size=2, max_basket_size=6)
logger.info(f"Generated {len(candidate_baskets)} candidate baskets")

# Test initial cointegration
logger.info("Testing initial cointegration on full 1-year window...")
cointegrated_baskets = []
for basket in tqdm(candidate_baskets[:1000], desc="Testing baskets"):  # Limit for speed
    result = test_basket_cointegration(price_data, basket)
    if result:
        cointegrated_baskets.append(result)

logger.info(f"✓ Found {len(cointegrated_baskets)} cointegrated baskets")


2025-10-31 11:10:43,024 - INFO - Generating candidate baskets...


In [ ]:
# Phase 4 & 5: Test persistence and spread constraints (OPTIMIZED - PARALLEL)
# Use parallel processing for speed (can be 10-100x faster depending on CPU cores)
logger.info("Testing persistence and spread constraints in parallel...")

# Option 1: Parallel (FAST - recommended)
USE_PARALLEL = True  # Set to False for sequential (slower but simpler debugging)

if USE_PARALLEL:
    import os
    max_workers = min(os.cpu_count() or 4, len(cointegrated_baskets))  # Use available CPU cores
    logger.info(f"Using {max_workers} parallel workers")
    valid_baskets = test_baskets_parallel(cointegrated_baskets, max_workers=max_workers)

logger.info(f"✓ {len(valid_baskets)} baskets passed persistence and spread constraints")


In [ ]:
# Phase 6: Optimize lookback windows
logger.info("Optimizing lookback windows...")
optimized_baskets = []

for basket_result in tqdm(valid_baskets, desc="Window optimization"):
    log_prices = basket_result['log_prices']
    eigenvector = basket_result['eigenvector']
    spread = basket_result['spread']
    basket_size = len(basket_result['basket'])
    
    opt_result = optimize_lookback_window(log_prices, eigenvector, spread, basket_size=basket_size)
    
    if opt_result['optimal_window']:
        basket_result['optimal_lookback'] = opt_result['optimal_window']
        basket_result['lookback_metrics'] = opt_result['optimal_metrics']
        basket_result['lookback_results'] = opt_result['all_results']
        optimized_baskets.append(basket_result)

logger.info(f"✓ Optimized {len(optimized_baskets)} baskets with valid lookback windows")


In [ ]:
# Phase 7: Optimize basket size and select best
opt_results = optimize_basket_size(optimized_baskets)
best_basket = opt_results['best_overall']
baskets_by_size = opt_results['best_by_size']

logger.info(f"\n✓ Best baskets by size:")
for size in sorted(baskets_by_size.keys()):
    best = baskets_by_size[size]
    logger.info(f"  Size {size}: {best['basket']} (score: {best['final_score']:.3f}, "
                f"persistence: {best['persistence']:.3f}, max_z: {best['max_zscore']:.2f})")

if best_basket:
    logger.info(f"\n✓ Best overall basket: {best_basket['basket']}")
    logger.info(f"  Score: {best_basket['final_score']:.3f}")
    logger.info(f"  Persistence: {best_basket['persistence']:.3f}")
    logger.info(f"  Max z-score: {best_basket['max_zscore']:.2f}")
    logger.info(f"  Optimal lookback: {best_basket['optimal_lookback']} days")
    logger.info(f"  Johansen p-value: {best_basket['johansen_result']['p_value']:.4f}")
else:
    logger.warning("No valid baskets found!")


In [ ]:
# Phase 8: Visualization and reporting
if best_basket:
    plot_spread_analysis(best_basket, price_data)
    plot_lookback_optimization(best_basket)
    print_summary_statistics(best_basket)
else:
    print("No valid baskets found. Consider:")
    print("1. Relaxing persistence threshold")
    print("2. Increasing max z-score limit")
    print("3. Testing more basket combinations")
